<a href="https://colab.research.google.com/github/RafaelCaballero/BME/blob/main/mfia/IBEXClas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# IBEX 35: clasificacion de Mapfre con validacion cruzada

El objetivo es predecir la clase del rendimiento de `mapfre_close` usando variables retardadas del IBEX, incluyendo Mapfre en dias anteriores. Las clases son:

- `0`: movimiento no significativo
- `1`: subida significativa
- `2`: bajada significativa

La metrica usada para elegir la mejor combinacion de columnas es la kappa de Cohen.


In [ ]:
try:
    import yfinance as yf
except ModuleNotFoundError:
    %pip install -q yfinance
    import yfinance as yf

import pandas as pd
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedKFold, cross_validate, cross_val_predict
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    cohen_kappa_score,
    confusion_matrix,
    make_scorer,
    precision_score,
    recall_score,
)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


## Valores del IBEX

La lista usa tickers de Yahoo Finance para los componentes del IBEX 35. Los nombres de la izquierda son los prefijos que tendran las columnas del dataframe final.


In [ ]:
IBEX_TICKERS = {
    "acs": "ACS.MC",
    "acerinox": "ACX.MC",
    "aena": "AENA.MC",
    "amadeus": "AMS.MC",
    "acciona": "ANA.MC",
    "accionaenergia": "ANE.MC",
    "bbva": "BBVA.MC",
    "bankinter": "BKT.MC",
    "caixabank": "CABK.MC",
    "cellnex": "CLNX.MC",
    "colonial": "COL.MC",
    "endesa": "ELE.MC",
    "enagas": "ENG.MC",
    "fluidra": "FDR.MC",
    "ferrovial": "FER.MC",
    "grifols": "GRF.MC",
    "iag": "IAG.MC",
    "iberdrola": "IBE.MC",
    "indra": "IDR.MC",
    "inditex": "ITX.MC",
    "logista": "LOG.MC",
    "mapfre": "MAP.MC",
    "merlin": "MRL.MC",
    "arcelormittal": "MTS.MC",
    "naturgy": "NTGY.MC",
    "puig": "PUIG.MC",
    "redeia": "RED.MC",
    "repsol": "REP.MC",
    "rovi": "ROVI.MC",
    "sabadell": "SAB.MC",
    "santander": "SAN.MC",
    "sacyr": "SCYR.MC",
    "solaria": "SLR.MC",
    "telefonica": "TEF.MC",
    "unicaja": "UNI.MC",
}


## Descarga y preparacion inicial

La funcion `descarga` baja el cierre ajustado y el volumen. Las columnas `*_close` son rendimientos porcentuales simples calculados como `100 * (precio_t / precio_t-1 - 1)`; las columnas `*_vol` conservan el volumen diario.


In [ ]:
def extrae_serie(datos, ticker, campo):
    """Devuelve una serie de yfinance tanto si el MultiIndex viene por ticker como por campo."""
    if not isinstance(datos.columns, pd.MultiIndex):
        return datos[campo]

    nivel_0 = datos.columns.get_level_values(0)
    nivel_1 = datos.columns.get_level_values(1)

    if ticker in nivel_0 and campo in nivel_1:
        return datos[(ticker, campo)]
    if campo in nivel_0 and ticker in nivel_1:
        return datos[(campo, ticker)]

    raise KeyError(f"No encuentro {campo} para {ticker}")


def descarga(fecha_inicial="2021-01-01", tickers=IBEX_TICKERS):
    """Descarga datos de Yahoo Finance y devuelve close pct-ret y volumen del IBEX."""
    lista_yahoo = list(tickers.values())
    datos = yf.download(
        lista_yahoo,
        start=fecha_inicial,
        auto_adjust=False,
        group_by="ticker",
        progress=False,
        threads=True,
    )

    columnas = {}
    avisos = []

    for nombre, ticker in tickers.items():
        try:
            cierre_ajustado = extrae_serie(datos, ticker, "Adj Close")
            volumen = extrae_serie(datos, ticker, "Volume")
        except KeyError as exc:
            avisos.append(str(exc))
            continue

        columnas[f"{nombre}_close"] = cierre_ajustado.pct_change() * 100
        columnas[f"{nombre}_vol"] = volumen

    df = pd.DataFrame(columnas).replace([np.inf, -np.inf], np.nan)
    columnas_vacias = df.columns[df.isna().all()].tolist()
    if columnas_vacias:
        print("Columnas sin datos en Yahoo Finance y eliminadas:")
        print(columnas_vacias)
        df = df.drop(columns=columnas_vacias)

    if avisos:
        print("Avisos de descarga:")
        for aviso in avisos:
            print("-", aviso)

    return df


In [ ]:
df = descarga("2021-01-01")
df.head()


In [ ]:
df.shape


## Variable objetivo de clasificacion

Dividimos el rendimiento de Mapfre en tres clases usando terciles. El tercio superior se considera subida significativa, el tercio inferior bajada significativa y el tercio central movimiento no significativo.


In [ ]:
CLASES = {
    0: "sin_movimiento_significativo",
    1: "subida_significativa",
    2: "bajada_significativa",
}


def clasifica_rendimiento(df, c="mapfre_close", metodo="terciles", umbral=None):
    if c not in df.columns:
        raise ValueError(f"La columna objetivo {c!r} no esta en df")

    r = df[c]

    if metodo == "terciles":
        limite_bajada = r.quantile(1 / 3)
        limite_subida = r.quantile(2 / 3)
    elif metodo == "umbral":
        if umbral is None:
            umbral = 0.005
        limite_bajada = -abs(umbral)
        limite_subida = abs(umbral)
    else:
        raise ValueError("metodo debe ser 'terciles' o 'umbral'")

    y_clase = pd.Series(0, index=df.index, name=f"{c}_clase")
    y_clase[r > limite_subida] = 1
    y_clase[r < limite_bajada] = 2

    return y_clase, limite_bajada, limite_subida


In [ ]:
c = "mapfre_close"
y_clase, limite_bajada, limite_subida = clasifica_rendimiento(df, c=c, metodo="terciles")

print(f"Limite bajada significativa: {limite_bajada:.6f}")
print(f"Limite subida significativa: {limite_subida:.6f}")
y_clase.value_counts().sort_index().rename(index=CLASES)


## Construccion de `df_p`

En cada fila dejamos la clase del dia actual y, como variables explicativas, todas las columnas retardadas de 1 a `p` dias, incluida `mapfre_close` de dias anteriores. No se usa `mapfre_close` del propio dia como predictor.


In [ ]:
def prepara_df_p_clasificacion(df, y_clase, c="mapfre_close", p=5):
    if c not in df.columns:
        raise ValueError(f"La columna objetivo {c!r} no esta en df")

    explicativas = df.copy()
    retardos = []

    for lag in range(1, p + 1):
        retardos.append(explicativas.shift(lag).add_suffix(f"_t-{lag}"))

    df_p = pd.concat([y_clase, *retardos], axis=1).dropna()
    df_p[y_clase.name] = df_p[y_clase.name].astype(int)
    return df_p


In [ ]:
p = 5
objetivo_clase = y_clase.name

df_p = prepara_df_p_clasificacion(df, y_clase, c=c, p=p)

columnas_prohibidas = [col for col in df_p.columns if col == c]
assert not columnas_prohibidas, f"Hay columnas sin retardo en df_p: {columnas_prohibidas}"
assert any(col.startswith(f"{c}_t-") for col in df_p.columns), "Faltan retardos de Mapfre"

df_p.head()


In [ ]:
df_p.shape


## Validacion cruzada para clasificacion

Usamos una regresion logistica multiclase con escalado previo. La seleccion de columnas se ordena por kappa de Cohen media en validacion cruzada repetida.


In [ ]:
def crea_modelo_clasificacion():
    return make_pipeline(
        StandardScaler(),
        LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            solver="lbfgs",
        ),
    )


def LogisticRegressionCV(X, y, repeticiones=100, divisiones=10, random_state=123):
    repartidor = RepeatedStratifiedKFold(
        n_splits=divisiones,
        n_repeats=repeticiones,
        random_state=random_state,
    )

    scoring = {
        "kappa": make_scorer(cohen_kappa_score),
        "accuracy": "accuracy",
        "precision_macro": make_scorer(precision_score, average="macro", zero_division=0),
        "recall_macro": make_scorer(recall_score, average="macro", zero_division=0),
    }

    cv_res = cross_validate(
        crea_modelo_clasificacion(),
        X,
        y,
        cv=repartidor,
        scoring=scoring,
        n_jobs=-1,
    )

    return {
        "kappa": cv_res["test_kappa"].mean(),
        "accuracy": cv_res["test_accuracy"].mean(),
        "precision_macro": cv_res["test_precision_macro"].mean(),
        "recall_macro": cv_res["test_recall_macro"].mean(),
    }


## Busqueda aleatoria de columnas

En cada iteracion se eligen `m` columnas al azar entre las variables retardadas. Se calcula la kappa de Cohen media y se guarda el resultado en `df_res`.


In [ ]:
n = 100
m = 5
repeticiones = 100
divisiones = 10
semilla = 123

XColumns = [col for col in df_p.columns if col != objetivo_clase]
y = df_p[objetivo_clase]

rng = np.random.default_rng(semilla)
resultados = []

for i in range(n):
    columnas_elegidas = list(rng.choice(XColumns, size=m, replace=False))
    X = df_p[columnas_elegidas]
    metricas = LogisticRegressionCV(
        X,
        y,
        repeticiones=repeticiones,
        divisiones=divisiones,
        #random_state=semilla,
    )

    resultados.append({
        "iteracion": i + 1,
        "columnas": columnas_elegidas,
        **metricas,
    })

    if (i + 1) % 10 == 0:
        mejor_kappa = max(r["kappa"] for r in resultados)
        print(f"Iteracion {i + 1}/{n}. Mejor kappa provisional: {mejor_kappa:.6f}")

df_res = pd.DataFrame(resultados).sort_values("kappa", ascending=False).reset_index(drop=True)
df_res


## Estadisticas de los experimentos

In [ ]:
kappa_mejor = df_res["kappa"].max()
kappa_peor = df_res["kappa"].min()
mejora_kappa = kappa_mejor - kappa_peor

if kappa_peor != 0:
    mejora_kappa_pct = 100 * mejora_kappa / abs(kappa_peor)
else:
    mejora_kappa_pct = np.nan

estadisticas_kappa = pd.DataFrame({
    "estadistico": ["mejor", "peor", "media", "mediana", "desviacion_tipica", "mejora_mejor_vs_peor", "mejora_mejor_vs_peor_%"],
    "valor": [
        kappa_mejor,
        kappa_peor,
        df_res["kappa"].mean(),
        df_res["kappa"].median(),
        df_res["kappa"].std(),
        mejora_kappa,
        mejora_kappa_pct,
    ],
})

estadisticas_kappa


## Mejor combinacion encontrada

In [ ]:
mejor = df_res.iloc[0]
mejores_columnas = mejor["columnas"]

print(f"Mejor kappa media: {mejor['kappa']:.6f}")
print("Columnas seleccionadas:")
for columna in mejores_columnas:
    print("-", columna)


## Matriz de confusion y metricas finales

Para la matriz de confusion usamos una validacion cruzada simple de `divisiones` particiones, de forma que cada fila tenga una unica prediccion fuera de muestra.


In [ ]:
X_mejor = df_p[mejores_columnas]
y = df_p[objetivo_clase]

cv_final = StratifiedKFold(
    n_splits=divisiones,
    shuffle=True,
    random_state=semilla,
)

y_pred = cross_val_predict(
    crea_modelo_clasificacion(),
    X_mejor,
    y,
    cv=cv_final,
    n_jobs=-1,
)

matriz_confusion = pd.DataFrame(
    confusion_matrix(y, y_pred, labels=[0, 1, 2]),
    index=[f"real_{CLASES[i]}" for i in [0, 1, 2]],
    columns=[f"pred_{CLASES[i]}" for i in [0, 1, 2]],
)

matriz_confusion


In [ ]:
metricas_finales = pd.DataFrame({
    "metrica": ["accuracy", "precision_macro", "recall_macro", "kappa_cohen"],
    "valor": [
        accuracy_score(y, y_pred),
        precision_score(y, y_pred, average="macro", zero_division=0),
        recall_score(y, y_pred, average="macro", zero_division=0),
        cohen_kappa_score(y, y_pred),
    ],
})

metricas_finales


In [ ]:
reporte_clasificacion = pd.DataFrame(
    classification_report(
        y,
        y_pred,
        labels=[0, 1, 2],
        target_names=[CLASES[i] for i in [0, 1, 2]],
        zero_division=0,
        output_dict=True,
    )
).T

reporte_clasificacion


In [ ]:
mejores_columnas
